# Aktivitet: Steinen fra broen — hvor mye betyr luftmotstanden?
**TRES0312 Fysikk — Kapittel 3: Kinematikk (Plenumsregning)**

I videoen slippes en stein fra en bro, og den bruker $t_0 = 2.4\,\mathrm{s}$ på å nå vannoverflaten.

I oppgaven regnet vi ut broens høyde ved å anta at steinen faller **uten luftmotstand**:

$$h_0 = \tfrac{1}{2} g t_0^2$$

Her undersøker vi hvor god denne antakelsen egentlig er, ved å regne **numerisk** på fallet med luftmotstand inkludert, og sammenligne.

**Antakelser om steinen** (ingen av disse er oppgitt i oppgaven — vi må gjette oss fram til noe rimelig): den er kuleformet med diameter $8\,\mathrm{cm}$ (radius $r=0.04\,\mathrm{m}$) og har en tetthet omtrent som vanlig stein/granitt, $\rho_{\text{stein}} \approx 2500\,\mathrm{kg/m^3}$. Luftens tetthet er $\rho_{\text{luft}} \approx 1.225\,\mathrm{kg/m^3}$, og luftmotstandskoeffisienten for en kule er typisk $C_d \approx 0.47$.

Luftmotstandskraften på en kule som beveger seg med fart $v$ er

$$F_L = \tfrac{1}{2}\,\rho_{\text{luft}}\,C_d\,A\,v^2, \qquad A = \pi r^2$$

og peker alltid motsatt av bevegelsesretningen (oppover, når steinen faller nedover).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -------------------------
# Parametere
# -------------------------
g = 9.81            # tyngdeakselerasjon [m/s^2]
t0 = 2.4             # observert falltid, uten luftmotstand antatt [s]
h0 = 0.5 * g * t0**2  # broens høyde ut fra a)

# Antakelser om steinen (se diskusjon over)
r = 0.04              # radius [m]
rho_stein = 2500.0    # tetthet [kg/m^3]
rho_luft = 1.225      # lufttetthet [kg/m^3]
Cd = 0.47             # luftmotstandskoeffisient for kule

V = 4/3 * np.pi * r**3
m = rho_stein * V               # masse [kg]
A = np.pi * r**2                # tverrsnittsareal [m^2]
k = 0.5 * rho_luft * Cd * A     # luftmotstandskonstant

print(f"Antatt høyde (uten luftmotstand): h0 = {h0:.2f} m")
print(f"Masse: m = {m:.3f} kg,  k = {k:.6f} kg/m")

In [ ]:
def simuler_fall(k, m, g=9.81, dt=1e-4, h_maks=100.0):
    """
    Simulerer et fritt fall med luftmotstand numerisk (Eulers metode).
    Bevegelsesligningen er m*dv/dt = m*g - k*v*|v|, dh/dt = v.
    Luftmotstanden er proporsjonal med v^2, men vi bruker v*|v| for at
    kraften alltid skal peke motsatt av bevegelsen (relevant om v<0).

    Returnerer tidspunktene, høyden falt, og farten ved hvert tidssteg.
    """
    t, v, h = 0.0, 0.0, 0.0
    ts, hs, vs = [t], [h], [v]
    while h < h_maks:
        a = g - (k / m) * v * abs(v)
        v += a * dt
        h += v * dt
        t += dt
        ts.append(t); hs.append(h); vs.append(v)
    return np.array(ts), np.array(hs), np.array(vs)

# Simuler helt til steinen har falt forbi h0, både med og uten luftmotstand
t_u, h_u, v_u = simuler_fall(k=0.0, m=m, h_maks=h0 * 1.05)   # uten luftmotstand
t_m, h_m, v_m = simuler_fall(k=k,   m=m, h_maks=h0 * 1.05)   # med luftmotstand

# Finn tidspunktet der hver kurve passerer h0 (interpolasjon)
t_stopp_u = np.interp(h0, h_u, t_u)
v_stopp_u = np.interp(h0, h_u, v_u)
t_stopp_m = np.interp(h0, h_m, t_m)
v_stopp_m = np.interp(h0, h_m, v_m)

print(f"Uten luftmotstand: bruker {t_stopp_u:.4f} s, treffer vannet i {v_stopp_u:.2f} m/s")
print(f"Med luftmotstand:  bruker {t_stopp_m:.4f} s, treffer vannet i {v_stopp_m:.2f} m/s")
print(f"Luftmotstanden øker falltiden med {(t_stopp_m - t_stopp_u)/t_stopp_u*100:.1f} %")
print(f"Luftmotstanden reduserer treffhastigheten med {(v_stopp_u - v_stopp_m)/v_stopp_u*100:.1f} %")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 4.2))

axs[0].plot(t_u, h_u, "--", color="gray", linewidth=2, label="Uten luftmotstand")
axs[0].plot(t_m, h_m, "-", color="royalblue", linewidth=2.5, label="Med luftmotstand")
axs[0].axhline(h0, color="black", linewidth=1, linestyle=":")
axs[0].set_xlabel("Tid [s]")
axs[0].set_ylabel("Fallhøyde [m]")
axs[0].set_title("Høyde falt")
axs[0].legend(fontsize=9)
axs[0].grid(True, alpha=0.3)

axs[1].plot(t_u, v_u, "--", color="gray", linewidth=2, label="Uten luftmotstand")
axs[1].plot(t_m, v_m, "-", color="tomato", linewidth=2.5, label="Med luftmotstand")
axs[1].set_xlabel("Tid [s]")
axs[1].set_ylabel("Fart [m/s]")
axs[1].set_title("Fart")
axs[1].legend(fontsize=9)
axs[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Konklusjon

For en stein på denne størrelsen og tettheten utgjør luftmotstanden bare en liten forskjell over et fall på under $30\,\mathrm{m}$ — falltiden øker med under $2\,\%$, og treffhastigheten reduseres med et par prosent. Dette henger sammen med at steinen er tung nok til at luftmotstanden er liten sammenlignet med tyngdekraften ved disse fartene (langt under steinens terminalfart). Antakelsen om null luftmotstand i a) er derfor god i dette tilfellet.

### Prøv selv
- Hva skjer med resultatet dersom steinen er mye lettere, f.eks. en liten isopor-kule med samme størrelse? Prøv å endre `rho_stein`.
- Hvor stor må falltiden/høyden være før luftmotstanden begynner å utgjøre mer enn f.eks. 10 %?